# Comparaison : Delta Hedging (Black-Scholes) vs Deep Hedging

**Référence** : Buehler, Gonon, Teichmann & Wood — *Deep Hedging* (arXiv 1802.03042, 2018)

---

## Objectif

Comparer deux approches de couverture d'un call européen **sans coûts de transaction** :

| Méthode | Description |
|---|---|
| **Delta Hedging (BS)** | Stratégie classique : $\delta_k = N(d_1)$, recalculé à chaque pas |
| **Deep – CVaR(α=0.50)** | Réseau de neurones, faible aversion au risque |
| **Deep – CVaR(α=0.95)** | Réseau de neurones, forte aversion au risque |
| **Deep – CVaR(α=0.99)** | Réseau de neurones, très forte aversion au risque |
| **Deep – Entropique(λ=1)** | Réseau de neurones, utilité exponentielle |

**Marché** : Black-Scholes (GBM), $S_0 = K = 100$, $\sigma = 20\%$, $T = 30$ jours, rebalancement journalier.

---

### Note sur l'architecture réseau

Le papier original utilise un réseau distinct par pas de temps. Ici, on utilise un **réseau à poids partagés** entre tous les pas, avec le temps restant $\tau = T - t_k$ comme feature explicite — ce qui est équivalent dans le cas Black-Scholes (la stratégie optimale est une fonction de $(\log(S_k/K),\, \tau)$). Cette simplification réduit le temps de compilation TensorFlow de 30× tout en conservant la même expressivité.

## 1. Imports et paramètres

In [ ]:
import os, math, time, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
warnings.filterwarnings("ignore")

import numpy as np
import tensorflow as tf
import keras
from keras import layers, ops
from scipy.stats import norm
import matplotlib.pyplot as plt

# ── Marché ────────────────────────────────────────────────────────────────────
S0      = 100.0        # prix initial
K       = 100.0        # strike (at-the-money)
SIGMA   = 0.20         # volatilité annuelle
T       = 30 / 365     # maturité : 30 jours
N_STEPS = 30           # rebalancement journalier

# ── Entraînement ─────────────────────────────────────────────────────────────
# Pour améliorer la qualité : augmenter N_TRAIN, N_EPOCHS
N_TRAIN  = 50_000      # trajectoires d'entraînement
N_TEST   = 100_000     # trajectoires hors-échantillon
N_HIDDEN = 30          # neurones cachés supplémentaires
BATCH    = 512         # mini-batch
N_EPOCHS = 60          # époques
LR       = 0.005       # taux d'apprentissage Adam

print(f"TensorFlow {tf.__version__}  |  Keras {keras.__version__}")

# Prix BS du call en t=0 (utilisé comme p₀ commun pour toutes les méthodes)
def bs_call_price(s, tau):
    tau = max(float(tau), 1e-10)
    d1  = (math.log(float(s) / K) + 0.5 * SIGMA**2 * tau) / (SIGMA * math.sqrt(tau))
    d2  = d1 - SIGMA * math.sqrt(tau)
    return float(s) * norm.cdf(d1) - K * norm.cdf(d2)

Q0 = bs_call_price(S0, T)
print(f"Prix risque-neutre du call en t=0 : q₀ = {Q0:.4f}")

## 2. Simulation Black-Scholes

Sous la mesure risque-neutre $\mathbb{Q}$ (taux $r = 0$) :
$$S_{t+dt} = S_t \exp\!\bigl(-\tfrac12\sigma^2 dt + \sigma\sqrt{dt}\,Z\bigr), \quad Z\sim\mathcal{N}(0,1)$$

In [ ]:
def simulate_bs(n_paths: int, seed: int = 0) -> np.ndarray:
    """Retourne les trajectoires GBM, shape (n_paths, N_STEPS+1)."""
    rng = np.random.default_rng(seed)
    dt  = T / N_STEPS
    dW  = rng.normal(0.0, math.sqrt(dt), (n_paths, N_STEPS))
    log_r = -0.5 * SIGMA**2 * dt + SIGMA * dW
    logS  = np.concatenate([
        np.full((n_paths, 1), math.log(S0)),
        math.log(S0) + np.cumsum(log_r, axis=1)
    ], axis=1)
    return np.exp(logS).astype(np.float32)


S_train = simulate_bs(N_TRAIN, seed=0)
S_test  = simulate_bs(N_TEST,  seed=999)   # seed différent → hors-échantillon

print(f"Train : {S_train.shape}  |  Test : {S_test.shape}")

fig, ax = plt.subplots(figsize=(9, 4))
for i in range(100):
    ax.plot(np.arange(N_STEPS + 1), S_train[i], lw=0.5, alpha=0.4)
ax.axhline(K, color="k", ls="--", lw=1.2, label=f"Strike K = {K}")
ax.set_xlabel("Jour"); ax.set_ylabel("$S_t$")
ax.set_title("100 trajectoires Black-Scholes (ensemble train)")
ax.legend(); plt.tight_layout(); plt.show()

## 3. Payoff et features réseau

In [ ]:
def call_payoff(S: np.ndarray) -> np.ndarray:
    """Z = max(S_T − K, 0), shape (n_paths,)."""
    return np.maximum(S[:, -1] - K, 0.0).astype(np.float32)


def make_features(S: np.ndarray) -> np.ndarray:
    """
    Features I_k = (log-moneyness, temps restant) aux dates t_0 … t_{n−1}.
    Shape : (n_paths, N_STEPS, 2).
    """
    times     = np.linspace(0, T, N_STEPS + 1)
    log_money = np.log(S[:, :-1] / K).astype(np.float32)   # (n, N_STEPS)
    tau_grid  = (T - times[:-1]).astype(np.float32)         # (N_STEPS,)
    tau_mat   = np.broadcast_to(tau_grid, log_money.shape)  # (n, N_STEPS)
    return np.stack([log_money, tau_mat], axis=-1)          # (n, N_STEPS, 2)


Z_train    = call_payoff(S_train)
Z_test     = call_payoff(S_test)
feat_train = make_features(S_train)
feat_test  = make_features(S_test)

print(f"Features shape : {feat_train.shape}")
print(f"Payoff moyen train : {Z_train.mean():.4f}  (doit ≈ q₀ = {Q0:.4f})")

## 4. Benchmark : Delta Hedging Black-Scholes classique

$$\delta^{\text{BS}}_k = N(d_1(S_k,\, T-t_k)), \qquad d_1 = \frac{\ln(S_k/K) + \tfrac12\sigma^2\tau}{\sigma\sqrt{\tau}}$$

$$\text{P\&L} = q_0 - Z + \sum_{k=0}^{n-1} \delta^{\text{BS}}_k \, (S_{k+1} - S_k)$$

In [ ]:
def bs_delta(s: np.ndarray, tau: float) -> np.ndarray:
    tau = max(tau, 1e-10)
    d1  = (np.log(s / K) + 0.5 * SIGMA**2 * tau) / (SIGMA * math.sqrt(tau))
    return norm.cdf(d1).astype(np.float32)


def bs_delta_hedge(S: np.ndarray) -> tuple:
    """Calcule (P&L, positions delta) sur un ensemble de trajectoires."""
    times    = np.linspace(0, T, N_STEPS + 1)
    holdings = np.zeros((S.shape[0], N_STEPS), dtype=np.float32)
    for k in range(N_STEPS):
        holdings[:, k] = bs_delta(S[:, k], T - times[k])

    Z             = call_payoff(S)
    trading_gains = np.sum(holdings * (S[:, 1:] - S[:, :-1]), axis=1)
    pnl           = (-Z + Q0 + trading_gains).astype(np.float32)
    return pnl, holdings


pnl_bs, delta_bs_h = bs_delta_hedge(S_test)

print("BS Delta Hedging (hors-échantillon)")
print(f"  Moyenne  : {pnl_bs.mean():+.4f}")
print(f"  Std      : {pnl_bs.std():.4f}")
cvar95_bs = float(np.mean(-pnl_bs[pnl_bs <= np.percentile(pnl_bs,  5)]))
cvar99_bs = float(np.mean(-pnl_bs[pnl_bs <= np.percentile(pnl_bs,  1)]))
print(f"  CVaR 95% : {cvar95_bs:.4f}")
print(f"  CVaR 99% : {cvar99_bs:.4f}")

## 5. Architecture du réseau Deep Hedging

**Structure semi-récurrente** (eq. 4.1 du papier) avec **poids partagés** :
$$\delta_k = F\bigl(\underbrace{\log(S_k/K),\; T-t_k}_{I_k},\; \delta_{k-1}\bigr), \quad \delta_{-1} = 0$$

Un seul réseau $F$ est partagé entre tous les pas de temps.  
Dans le modèle BS, la stratégie optimale est une fonction univoque de $(I_k, \delta_{k-1})$, donc cette simplification est exacte.

In [ ]:
D_FEAT = 2   # (log-moneyness, tau)


def build_network() -> keras.Model:
    """
    MLP semi-récurrent à poids partagés.
    Entrée : (log(S/K), tau, delta_prev)  →  Sortie : delta_k
    """
    hidden   = D_FEAT + N_HIDDEN   # taille des couches cachées
    step_net = keras.Sequential([
        layers.Dense(hidden, activation="relu"),
        layers.Dense(hidden, activation="relu"),
        layers.Dense(1),
    ], name="step_net")

    feat_in    = keras.Input(shape=(N_STEPS, D_FEAT), name="features")
    delta_prev = feat_in[:, 0, :1] * 0.0   # δ_{-1} = 0

    deltas = []
    for k in range(N_STEPS):
        I_k     = feat_in[:, k, :]                            # (batch, 2)
        x       = ops.concatenate([I_k, delta_prev], axis=1)  # (batch, 3)
        delta_k = step_net(x)                                 # (batch, 1)
        deltas.append(delta_k)
        delta_prev = delta_k

    output = ops.stack(deltas, axis=1)   # (batch, N_STEPS, 1)
    return keras.Model(inputs=feat_in, outputs=output, name="deep_hedge")


_m = build_network()
print(f"Paramètres totaux : {_m.count_params():,}")
print(f"Sortie test : {_m(tf.zeros((4, N_STEPS, D_FEAT))).shape}")

## 6. Mesures de risque convexes (cadre OCE)

**Mesure OCE** (eq. 3.5) : $\;\rho(X) = \inf_{w\in\mathbb{R}}\bigl\{ w + \mathbb{E}[\ell(-X-w)]\bigr\}$

La variable $w$ est **co-optimisée** avec les poids du réseau.

| Mesure de risque | $\ell(x)$ | Lien avec la littérature |
|---|---|---|
| CVaR(α) | $\max(x,0)/(1-\alpha)$ | Expected Shortfall, Ex. 3.9 |
| Entropique(λ) | $e^{\lambda x} - (1+\log\lambda)/\lambda$ | Utilité exponentielle, Ex. 3.8 |

In [ ]:
class OCERisk:
    """Mesure de risque OCE avec variable w entraînable (Section 3.2)."""

    def __init__(self, loss_fn, name: str):
        self.loss_fn = loss_fn
        self.w       = keras.Variable(0.0, trainable=True, dtype=tf.float32)
        self.name    = name

    def __call__(self, pnl: tf.Tensor) -> tf.Tensor:
        return self.w + ops.mean(self.loss_fn(-pnl - self.w))


def cvar_risk(alpha: float) -> OCERisk:
    """CVaR / Expected Shortfall au niveau α (Exemple 3.9)."""
    def loss(x):
        return ops.maximum(x, 0.0) / (1.0 - alpha)
    return OCERisk(loss, name=f"CVaR(α={alpha:.2f})")


def entropic_risk(lam: float = 1.0) -> OCERisk:
    """Mesure entropique = utilité exponentielle (Exemple 3.8)."""
    c = (1.0 + math.log(lam)) / lam
    def loss(x):
        return ops.exp(lam * x) - c
    return OCERisk(loss, name=f"Entropique(λ={lam})")


# Les 4 mesures à comparer
RISK_CONFIGS = [
    ("CVaR(α=0.50)",     lambda: cvar_risk(0.50)),
    ("CVaR(α=0.95)",     lambda: cvar_risk(0.95)),
    ("CVaR(α=0.99)",     lambda: cvar_risk(0.99)),
    ("Entropique(λ=1)",  lambda: entropic_risk(1.0)),
]
print("Mesures configurées :", [n for n, _ in RISK_CONFIGS])

## 7. Boucle d'entraînement

P&L terminal (eq. 2.1, sans coûts) :
$$\text{PLT} = -Z + q_0 + \sum_{k=0}^{n-1} \delta_k \, (S_{k+1} - S_k)$$

Optimisation : descente de gradient Adam sur mini-batches (Section 4.3 / 5.1).

In [ ]:
Q0_tf = tf.constant(Q0, dtype=tf.float32)


def train_model(model:   keras.Model,
                risk:    OCERisk,
                verbose: bool = True) -> list:
    """Entraînement Adam avec train_step compilé JIT."""
    optimizer = keras.optimizers.Adam(learning_rate=LR)
    trainable = model.trainable_variables + [risk.w]

    # Pré-charger les données en mémoire GPU/CPU une seule fois
    S_tf    = tf.constant(S_train)
    feat_tf = tf.constant(feat_train)
    Z_tf    = tf.constant(Z_train)

    # Compilation JIT du step complet (gradient + mise à jour)
    @tf.function(reduce_retracing=True)
    def train_step(feat_b, S_b, Z_b):
        with tf.GradientTape() as tape:
            delta         = ops.squeeze(model(feat_b, training=True), axis=-1)  # (b, n)
            trading_gains = ops.sum(delta * (S_b[:, 1:] - S_b[:, :-1]), axis=1)
            pnl           = -Z_b + Q0_tf + trading_gains
            loss          = risk(pnl)
        grads = tape.gradient(loss, trainable)
        optimizer.apply_gradients(zip(grads, trainable))
        return loss

    rng     = np.random.default_rng(42)
    history = []

    for epoch in range(N_EPOCHS):
        perm   = rng.permutation(N_TRAIN)
        losses = []
        for i in range(0, N_TRAIN, BATCH):
            idx = perm[i : i + BATCH]
            loss = train_step(
                tf.gather(feat_tf, idx),
                tf.gather(S_tf, idx),
                tf.gather(Z_tf, idx)
            )
            losses.append(float(loss))

        avg = float(np.mean(losses))
        history.append(avg)
        if verbose and (epoch % 20 == 0 or epoch == N_EPOCHS - 1):
            print(f"    Ép. {epoch+1:3d}/{N_EPOCHS}  loss={avg:+.4f}  w={float(risk.w):+.4f}")

    return history


print("Boucle d'entraînement définie.")

## 8. Entraînement de tous les modèles Deep Hedging

In [ ]:
trained_models   = {}   # name  →  keras.Model
training_history = {}   # name  →  list[float]

t_total = time.time()

for name, risk_factory in RISK_CONFIGS:
    print(f"\n{'─'*55}")
    print(f"  Modèle : {name}")
    print(f"{'─'*55}")

    model = build_network()
    risk  = risk_factory()
    t0    = time.time()

    history = train_model(model, risk)

    elapsed = time.time() - t0
    print(f"  → {elapsed:.0f}s  |  loss finale = {history[-1]:+.4f}")
    trained_models[name]   = model
    training_history[name] = history

print(f"\n✓ Terminé en {time.time()-t_total:.0f}s au total.")

## 9. Évaluation hors-échantillon

In [ ]:
def eval_pnl(model: keras.Model) -> np.ndarray:
    """P&L sur le jeu de test (hors-échantillon)."""
    feat = tf.constant(feat_test)
    S    = tf.constant(S_test)
    Z    = tf.constant(Z_test)
    d    = ops.squeeze(model(feat, training=False), axis=-1)
    pnl  = -Z + Q0_tf + ops.sum(d * (S[:, 1:] - S[:, :-1]), axis=1)
    return np.array(pnl, dtype=np.float32)


def risk_stats(pnl: np.ndarray) -> dict:
    """Statistiques de risque."""
    losses = -pnl
    return {
        "Moy. P&L":  float(pnl.mean()),
        "Std P&L":   float(pnl.std()),
        "CVaR 95%":  float(np.mean(losses[losses >= np.percentile(losses, 95)])),
        "CVaR 99%":  float(np.mean(losses[losses >= np.percentile(losses, 99)])),
        "P&L min":   float(pnl.min()),
    }


# Collecter tous les P&L
pnl_results = {"BS Delta (classique)": pnl_bs}
for name, model in trained_models.items():
    pnl_results[f"Deep – {name}"] = eval_pnl(model)

# Affichage tableau
print(f"\n{'Méthode':<35s} {'Moy.':>8s} {'Std':>8s} {'CVaR95%':>9s} {'CVaR99%':>9s} {'min':>8s}")
print("─" * 82)
for label, pnl in pnl_results.items():
    s = risk_stats(pnl)
    print(f"{label:<35s} {s['Moy. P&L']:>+8.4f} {s['Std P&L']:>8.4f} "
          f"{s['CVaR 95%']:>9.4f} {s['CVaR 99%']:>9.4f} {s['P&L min']:>+8.4f}")

## 10. Courbes de convergence

In [ ]:
COLORS = ["steelblue", "darkorange", "forestgreen", "crimson"]

fig, ax = plt.subplots(figsize=(9, 4))
for (name, hist), color in zip(training_history.items(), COLORS):
    ax.plot(range(1, N_EPOCHS + 1), hist, lw=1.8, label=name, color=color)
ax.set_xlabel("Époque")
ax.set_ylabel("Mesure de risque OCE (objectif)")
ax.set_title("Convergence de l'entraînement")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 11. Distributions du P&L — panneaux individuels (cf. Figure 2 du papier)

In [ ]:
all_items = list(pnl_results.items())
palette   = ["dimgray"] + COLORS
bins      = np.linspace(-2.5, 2.5, 80)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes      = axes.flatten()

for i, ((label, pnl), color) in enumerate(zip(all_items, palette)):
    ax = axes[i]
    ax.hist(pnl, bins=bins, color=color, alpha=0.75, edgecolor="none")
    ax.axvline(0,          color="black", lw=0.8, ls="--", alpha=0.5)
    ax.axvline(pnl.mean(), color="red",   lw=1.2, label=f"μ={pnl.mean():+.3f}")
    s = risk_stats(pnl)
    ax.text(0.97, 0.97,
            f"σ = {s['Std P&L']:.3f}\nCVaR95 = {s['CVaR 95%']:.3f}\nCVaR99 = {s['CVaR 99%']:.3f}",
            transform=ax.transAxes, ha="right", va="top", fontsize=8.5,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.85))
    ax.set_title(label, fontsize=10); ax.set_xlabel("P&L"); ax.set_ylabel("Fréquence")
    ax.legend(fontsize=8); ax.grid(alpha=0.25)

axes[-1].set_visible(False)
fig.suptitle("Distribution du P&L terminal — sans coûts de transaction (hors-échantillon)",
             fontsize=12, y=1.01)
plt.tight_layout(); plt.show()

## 12. Densités superposées

In [ ]:
bins2 = np.linspace(-3.5, 3.5, 120)

fig, ax = plt.subplots(figsize=(10, 5))
for (label, pnl), color in zip(all_items, palette):
    counts, edges = np.histogram(pnl, bins=bins2, density=True)
    centers       = 0.5 * (edges[:-1] + edges[1:])
    lw = 2.5 if label == "BS Delta (classique)" else 1.6
    ls = "-"  if label == "BS Delta (classique)" else "--"
    ax.plot(centers, counts, lw=lw, ls=ls, color=color, label=label)

ax.axvline(0, color="black", lw=0.8, ls=":", alpha=0.6)
ax.set_xlabel("Erreur de couverture terminale (P&L)")
ax.set_ylabel("Densité")
ax.set_title("Comparaison des distributions — BS Delta vs Deep Hedging\n"
             "(sans coûts de transaction, évaluation hors-échantillon)")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 13. Stratégie delta apprise vs BS analytique (cf. Figure 3 du papier)

Comparaison des positions $\delta_k$ en fonction du spot $S_k$, à mi-maturité ($\tau = T/2$).

In [ ]:
tau_mid   = T / 2
k_mid     = N_STEPS // 2
spot_grid = np.linspace(75, 125, 300).astype(np.float32)

delta_bs_analytic = bs_delta(spot_grid, tau_mid)

# Features : log-moneyness et tau sur la grille, au pas k_mid
feat_grid         = np.zeros((len(spot_grid), N_STEPS, D_FEAT), dtype=np.float32)
feat_grid[:, k_mid, 0] = np.log(spot_grid / K)
feat_grid[:, k_mid, 1] = tau_mid

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(spot_grid, delta_bs_analytic, "k-", lw=2.8,
        label="BS Delta (analytique)", zorder=6)

for (name, model), color in zip(trained_models.items(), COLORS):
    out   = np.array(ops.squeeze(model(tf.constant(feat_grid), training=False), axis=-1))
    delta = out[:, k_mid]
    ax.plot(spot_grid, delta, lw=1.8, ls="--", color=color,
            label=f"Deep – {name}", alpha=0.90)

ax.axvline(K, color="gray", ls=":", lw=1.2, label="Strike K")
ax.set_xlabel("Prix du sous-jacent $S_t$")
ax.set_ylabel("Position $\\delta$ (unités)")
ax.set_title(f"Stratégie delta à mi-maturité ($\\tau = T/2 = {tau_mid:.4f}$ an)")
ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_ylim(-0.05, 1.05)
plt.tight_layout(); plt.show()

## 14. Tableau de synthèse final

In [ ]:
import pandas as pd

rows = []
for label, pnl in pnl_results.items():
    s = risk_stats(pnl)
    rows.append({
        "Méthode":          label,
        "Moy. P&L":         round(s["Moy. P&L"],  4),
        "Std P&L":          round(s["Std P&L"],    4),
        "CVaR 95% (perte)": round(s["CVaR 95%"],   4),
        "CVaR 99% (perte)": round(s["CVaR 99%"],   4),
        "P&L min":          round(s["P&L min"],     4),
    })

df = pd.DataFrame(rows).set_index("Méthode")

def highlight_best(col):
    is_loss = "CVaR" in col.name or "min" in col.name
    best    = col == (col.min() if is_loss else col.max())
    return ["background-color: #c8f7c5" if v else "" for v in best]

df.style.apply(highlight_best)

## 15. Résumé et interprétation

### Résultats attendus

| Observation | Explication mathématique |
|---|---|
| **CVaR(α=0.50) ≈ BS Delta** | CVaR→ espérance quand α→0. Le réseau converge vers la réplication parfaite (marché BS complet, Lemme 3.3) |
| **CVaR(α=0.99) : moins de pertes extrêmes** | La forte aversion au risque pousse le réseau à sacrifier la variance globale pour réduire la queue gauche |
| **Entropique : distribution plus resserrée** | La pénalité exponentielle $e^{\lambda x}$ rend le réseau très sensible aux grandes pertes |
| **BS Delta : P&L centré en 0, std faible** | Dans un marché BS sans frictions, la couverture delta est quasi-parfaite en temps discret |

### Avantages déjà visibles sans coûts de transaction

- **Flexibilité** : une seule méthode couvre toute la gamme de préférences de risque via le choix de $\rho$
- **Sans grecs** : le réseau apprend directement depuis les trajectoires simulées
- **Convergence garantie** : Proposition 4.9 — $\pi^M(X) \xrightarrow{M\to\infty} \pi(X)$

### Ce qui change avec des coûts de transaction

Avec $\varepsilon > 0$, le **delta BS sur-trade** (il ignore les coûts dans son calcul), tandis que le réseau deep hedging les **intègre nativement** dans l'objectif d'entraînement. Le deep hedging réduit le volume de trading et améliore le P&L net des frictions — c'est là son avantage décisif sur le delta classique.